# Goal

Исслед. `35.4c` показало, что важно выбирать родителей, которые несут полезный навык. Теперь интересно посмотреть - насколько важен детерминизм при этом. Т.е. повторяем `35.4c`, но с включенным детерминизмом

In [1]:
import os
os.environ['DOCKER_REGISTRY'] = 'develorium'

# TARGET_NOTEBOOK_FNAME

In [2]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_07.ipynb'

# OPTUNA_DECISIVE_METRIC


In [3]:
OPTUNA_DECISIVE_METRIC = 'test/levels_passed_mean'

# GRID_SEARCH_SPACE

In [4]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [5]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True # ВКЛЮЧАЕМ!!
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    HP.general.is_torch_amp_on_rollout = False
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    HP.vision_head.parent = dict(model='18d_world_model_09:183', weights='18d_world_model_09:183')
    HP.vision_head.is_trainable = False

    parent = optuna_trial.suggest_categorical('parent', [
        '18n_ppo_tr_frostbite_07:984',
        '18n_ppo_tr_frostbite_07:987',
    ])
    HP.encoder.parent = dict(model='18d_world_model_09:183', weights=parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = parent 
    HP.agent.sequence_length = 4
    HP.agent.action_plan_length = 10 
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.env_rams = None
    HP.test.env_ram_patches = None
    HP.test.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = None
    HP.ppo.rollout_env_ram_patches = [
        ['nine_lives'], 
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 512 
    HP.ppo.learn_rate = 'const(0.00015)'
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.is_obs_usage_enforced = True # ИДЁМ ОТ ОБЗОВ В ПРИНУДИТЕЛЬНОМ ПОРЯДКЕ, НЕСМОТРЯ НА ЗАМОРОЖЕННЫЙ VisionHead

    HP.ppo.ob_features_dropout = 0
    HP.ppo.ob_features_gauss_noise = 0
    
    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef='const(0.03)')
    HP.ppo.consistency_loss = dict(type='MSE', coef=0.1)
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    
    return HP

# Results


По решающей метрике **детерминизм не играет роли**: `35.4d` (детерминизм ВКЛ) и `35.4c` (детерминизм ВЫКЛ) не отличаются — средние `test/levels_passed_mean` **совпадают** (3.6797), Welch p=1.00, MWU p=0.91. По роллауту (`rollout/episode_lp_mean`) разница тоже не значима (Welch p=0.45, MWU p=0.44). Единственный содержательный намёк — «пролом потолка» (`test/levels_passed_max ≥ 5`) держится хуже: **7/12 у `35.4d` против 11/12 у `35.4c`** (Fisher p=0.155 — в сторону `35.4c`, но недобор мощности).

Оба ноутбука отличаются **ровно одним** HP: `is_torch_deterministic` (`False` у `35.4c`, `True` у `35.4d`). Остальное байт-в-байт одинаково: родители `07:984, :987`, `lr=const(0.00015)`, `ent=const(0.03)`, `prediction_loss=0.1`, `is_obs_usage_enforced=True`, rollout patches `[['nine_lives']]`, VH `:183` (frozen), `seq=4`.

## Основные метрики

|что|`18n_study_35.4d` (детерминизм ВКЛ)|`18n_study_35.4c` (детерминизм ВЫКЛ)|
|---|---|---|
|episode_r|<img src="./img/episode_r_35.4d.png">|<img src="./img/episode_r_35.4c.png">|
|episode_lp|<img src="./img/episode_lp_35.4d.png">|<img src="./img/episode_lp_35.4c.png">|
|levels_passed|<img src="./img/levels_passed_35.4d.png">|<img src="./img/levels_passed_35.4c.png">|
|levels_passed_mean|<img src="./img/levels_passed_mean_35.4d.png">|<img src="./img/levels_passed_mean_35.4c.png">|

## Статистика: `35.4c` vs `35.4d`

### a) `test/levels_passed_mean` (решающая)

|метрика|`18n_study_35.4d`|`18n_study_35.4c`|
|---|---|---|
|Sample Size (n)|12|12|
|Mean Score|3.6797|3.6797|
|Best Value (model-trial)|4.3125 (`:1146`)|4.2188 (`:1144`)|
|Standard Deviation|0.4495|0.3231|
|Shapiro-Wilk (p)|0.208|0.334|
|Welch's t-test (p)|1.000|—|
|Mann-Whitney U (p)|0.908|—|
|Cohen's d|0.000|—|
|**Вердикт**|**н/з** (p ≥ 0.05; эффект отсутствует)||

### b) `rollout/episode_lp_mean` (среднее по рану)

|метрика|`18n_study_35.4d`|`18n_study_35.4c`|
|---|---|---|
|Sample Size (n)|12|12|
|Mean Score|4.1746|4.3221|
|Best Value|4.9101|4.8460|
|Standard Deviation|0.4920|0.4389|
|Shapiro-Wilk (p)|0.056|0.112|
|Welch's t-test (p)|0.447|—|
|Mann-Whitney U (p)|0.436|—|
|Cohen's d|0.316 (малый)|—|
|**Вердикт**|**н/з** (p ≥ 0.05; малый эффект)|—|

MDE (независимые выборки, n=12/руку, 80% мощности, α=0.05): `levels_passed_mean` ≈ 0.47, `episode_lp_mean` ≈ 0.56. Наблюдаемые Δ средних: 0.000 и −0.148 ⇒ оба **много ниже MDE**, т.е. тест не увидел бы и среднего эффекта. Разложение шума (decisive): межгрупповая дисперсия 0.000 vs внутригрупповая 0.153 — весь разброс от сида, не от режима детерминизма.

**Носители навыка 5-го уровня** (потолок мутированной эры): `test/levels_passed_max ≥ 5` — `35.4c` 11/12 (max 6), `35.4d` 7/12 (max 6), Fisher p=0.155; роллаут `episode_lp_max ≥ 5` — 12/12 vs 11/12 (p=1.0). Обе руки пробили потолок 4.

**Выводы**
1) **Детерминизм не влияет на решающую метрику**: `35.4c` и `35.4d` не отличаются ни по `levels_passed_mean` (p=1.00/0.91), ни по `episode_lp_mean` (p=0.45/0.44). Совпадение средних до 4 знаков (3.6797) — численная случайность (одинаковая сумма 44.15625 при разных распределениях), а не общий эффект.
2) **Роль детерминизма, поднятая в `35.4c`, в решающей метрике не подтверждается.** Но есть слабый сигнал по глубине: у детерминированного прогона «пролом потолка» (≥5 уровней на тесте) держится реже (7/12 vs 11/12), хотя и незначимо при n=12. Если этот трейт важен, нужна репликация с большим n (MDE≈0.47 — не хватает мощности на малый эффект).
3) Наблюдение по селекции родителей (из `35.4c`) сохраняется: выбор по `episode_lp` даёт носителей, а `levels_passed_mean` рук почти идентичен — решающая метрика слепа к навыку 5-го уровня.
4) **Статус системной проблемы**: не проверено по журналу (журнал несёт только решающую метрику).


# System

In [6]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [7]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_07',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_35.4d/18n_study_35.4d.ipynb',
 'optuna_study_name': '18n_study_35.4d',
 'optuna_study_serial': '35.4d',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_35.4d/18n_study_35.4d.optuna'}

In [8]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [9]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [10]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC=OPTUNA_DECISIVE_METRIC,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [11]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')

    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [12]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 12
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-20 13:51:43,869] A new study created in Journal with name: 18n_study_35.4d


2026.10.20-13:51:44.334439     0.442 >> Model instance registered, version=1145
2026.10.20-13:51:44.352530     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch1145.ipynb"
2026.10.20-13:51:44.353391     0.007 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:1145"; running launches=1
2026.10.20-13:52:15.423956     0.392 >> Model instance registered, version=1146
2026.10.20-13:52:15.437321     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch1146.ipynb"
2026.10.20-13:52:15.438011     0.005 >> 5.7 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:1146"; running launches=2
2026.10.20-13:52:47.207164     0.334 >> Model instance registered, version=1147
2026.10.20-13:52:47.222105     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch1147.ipynb"
2026.10.20-13:52:47.222618     0.007 >> 4.7 idle runners exist, submitted launch "18n_ppo_tr_fros

In [13]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-20 15:42:19,275] Using an existing study with name '18n_study_35.4d' instead of creating a new one.


2026.10.20-15:42:19.278924     5.035 >> Study statistics: 
2026.10.20-15:42:19.283743     0.005 >> 	Number of finished trials: 12
2026.10.20-15:42:19.284887     0.001 >> 	Number of pruned trials: 0
2026.10.20-15:42:19.285722     0.001 >> 	Number of complete trials: 12
2026.10.20-15:42:19.286765     0.001 >> Best trial:
2026.10.20-15:42:19.288154     0.001 >> 	Value: 4.3125
2026.10.20-15:42:19.288731     0.001 >> 	Model version: 1146
2026.10.20-15:42:19.289174     0.000 >> 	Params: 
2026.10.20-15:42:19.289753     0.001 >> 		parent: 18n_ppo_tr_frostbite_07:987


In [14]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.20-15:42:19.296615     0.007 >> Top trials:
2026.10.20-15:42:19.297702     0.001 >> 1) 18n_ppo_tr_frostbite_07:1146, value=4.3125
2026.10.20-15:42:19.298088     0.000 >> 2) 18n_ppo_tr_frostbite_07:1148, value=4.15625
2026.10.20-15:42:19.298576     0.000 >> 3) 18n_ppo_tr_frostbite_07:1155, value=4.0


In [15]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(3.6796875), np.float64(0.43039714127622875))